# Topic Modelling

**Latest version:** May 2026

## Notebook overview

This notebook fits the BERTopic model on the full corpus using the embeddings and optimal parameters produced by `2_embeddings_gridsearch.ipynb`.

| Step | What happens | Output |
|------|-------------|--------|
| 1 | Load packages and user settings | - |
| 2 | Load inputs | - |
| 3 | Define functions for BERTopic modelling | - |
| 4 | Fit BERTopic on the full corpus& reduce outliers | `topic_model_final`, `topics_final.npy`, `topic_info.csv`, `topic_info.xlsx`, `bertopic_results_final.csv.gz` |

**⚠️ Runtime note:** Running BERTopic in Step 3 can a long time (hours on large corpora). The model is saved after each major step so the notebook can be safely interrupted and resumed from the reload cells.  

**🔁 Reproducibility note:** `random_seed=42` is used throughout. Results may vary slightly across hardware due to UMAP's approximate nearest-neighbour algorithm.

---

# 1. Imports & configuration

In [1]:
from config import *
import gc
import json
import os
import pickle
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import silhouette_score
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
import nltk
from nltk.corpus import stopwords

# Download stopwords
nltk.download('stopwords', quiet=True)

# Environment optimization: reduce thread contention between native libraries
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]       = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Ensure output directories exist
for d in [TOPIC_MODEL_DIR, TOPIC_INFO_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

## ⚙️ User settings

**This is the only cell you need to edit before running the notebook.**

| Parameter | What it does |
|-----------|-------------|
| `DESIRED_TRAINING_SIZE` | Maximum number of documents used to *train* the model. The full corpus is always used for topic assignment. If your corpus is smaller than this value it is used in full automatically. |
| `BATCH_SIZE` | Documents processed per batch during the prediction phase. Reduce if you run into memory errors. |
| `RANDOM_SEED` | Seed for all random operations (shuffling, UMAP). |

In [5]:
# # 🔧 Edit these values before running
DESIRED_TRAINING_SIZE = 350_000
BATCH_SIZE = 50_000
RANDOM_SEED = 42

---
# 2. Load inputs

Three inputs are required from the previous notebooks:
1. **`docs.pkl`**: preprocessed document list from `1_data_preparation.ipynb`
2. **`final_embeddings.npy`**: full-corpus embeddings from `2_embeddings_gridsearch.ipynb`
3. **`best_params.json`**: optimal UMAP/HDBSCAN parameters from `2_embeddings_gridsearch.ipynb`

In [ ]:
# Documents
print("Loading preprocessed documents...")
with open(PREPROCESSED_DOCS_PATH, "rb") as f:
    preprocessed_docs = pickle.load(f)

# Generate positional IDs (0, 1, 2, ...) – used when linking back to metadata
preprocessed_ids = list(range(len(preprocessed_docs)))

print(f"✅ Loaded {len(preprocessed_docs):,} documents")

In [ ]:
# Embeddings (memory-mapped)
embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")

print(f"✅ Embeddings shape : {embeddings.shape}")
print(f"   dtype            : {embeddings.dtype}")
print(f"   file size        : {embeddings.nbytes / 1e9:.2f} GB")
assert embeddings.shape[0] == len(preprocessed_docs), (
    f"❌ Mismatch: {embeddings.shape[0]:,} embeddings vs {len(preprocessed_docs):,} docs"
)

In [ ]:
# Best parameters
if Path(BEST_PARAMS_PATH).exists():
    with open(BEST_PARAMS_PATH) as f:
        run_meta = json.load(f)
    best_params = run_meta["best_params"]
    print(f"✅ Loaded best_params from JSON")
    print(f"   Embedding model  : {run_meta['provenance']['embedding_model']}")
    print(f"   Grid search size : {run_meta['provenance']['gridsearch_sample_size']:,}")
else:
    print("⚠️  best_params.json not found – falling back to CSV")
    grid_results_df = pd.read_csv(FINAL_CSV_PATH_GRIDSEARCH)
    best_params = grid_results_df.iloc[0].to_dict()

print(f"\nParameters to use:")
for k, v in best_params.items():
    print(f"   {k}: {v}")

## Pre-flight checklist

Run the cell below to confirm all inputs are aligned before starting the long-running model fit.

In [ ]:
ACTUAL_TRAINING_SIZE = min(DESIRED_TRAINING_SIZE, len(preprocessed_docs))

print("Pre-flight checklist:")
print(f"  ✓ Documents loaded       : {len(preprocessed_docs):,}")
print(f"  ✓ Embeddings loaded      : {embeddings.shape}")
print(f"  ✓ Docs / embeddings match: {len(preprocessed_docs) == embeddings.shape[0]}")
print(f"  ✓ Best params loaded     : {bool(best_params)}")
print()
print("Run settings:")
print(f"  Desired training size : {DESIRED_TRAINING_SIZE:,}")
print(f"  Actual training size  : {ACTUAL_TRAINING_SIZE:,}",
      "(full corpus)" if ACTUAL_TRAINING_SIZE < DESIRED_TRAINING_SIZE else "")
print(f"  Batch size            : {BATCH_SIZE:,}")
print(f"  Random seed           : {RANDOM_SEED}")

---
# 3. BERTopic functions

### `fit_bertopic`

Trains BERTopic on a randomly shuffled subset of the corpus (`training_sample_size`) to discover topics, then assigns topics to all remaining documents in batches via `transform`.

| Argument | Type | Description |
|----------|------|-------------|
| `docs` | `list[str]` | Full document list |
| `embeddings` | `np.ndarray` | Pre-computed embeddings (same order as `docs`) |
| `best_params` | `dict` | Keys: `n_neighbors`, `n_components`, `min_dist`, `min_cluster_size`, `min_samples` |
| `batch_size` | `int` | Docs per batch during prediction (default 50 k) |
| `training_sample_size` | `int` | Docs used for training (default 350 k; auto-capped to corpus size) |
| `random_seed` | `int` | Seed for shuffling and UMAP (default 42) |

Returns `(topic_model, topics)` where `topics` is a `np.ndarray` in **original document order**.

**📍 Note:**
- Currently using stopwords from `nltk.corpus`, empty the `languages` list if you do not wish to use them
- Add your own stopwords in `custom_stops`
- In `BERTopic(calculate_probabilities)` is set to a default of `False` for faster modelling. Enable this if you wish to calculate the probability of the assigned topic per document. It is often unnecessary unless you need soft clustering, uncertainty analysis, downstream weighting. Please note that it will significantly increase runtime if enabled.

In [ ]:
def fit_bertopic(
    docs,
    embeddings,
    best_params,
    batch_size=50_000,
    training_sample_size=350_000,
    random_seed=42,
):
    print(f"Preparing {len(docs):,} documents for representative sampling...")
    start_time = time.time()

    np.random.seed(random_seed)
    shuffle_indices = np.random.permutation(len(docs))
    print(f"  ✓ Indices generated in {time.time() - start_time:.2f}s")

    actual_training_size = min(training_sample_size, len(docs))
    print(f"Training on {actual_training_size:,} documents "
          f"({actual_training_size / len(docs) * 100:.1f}% of corpus)")

    # Vectorizer, using multilingual stopwords
    languages = ["english", "dutch", "spanish", "french", "german"]
    multilingual_stops = []
    for lang in languages:
        multilingual_stops.extend(stopwords.words(lang))

    ########## Add custom stopwords here ##########
    custom_stops = [
        'and', 'of', 'the'
    ]
    ##########
    
    all_stops = list(set(multilingual_stops + custom_stops))
    print(f"  ✓ Loaded {len(all_stops):,} stopwords across {len(languages)} languages")

    # Model initialization, using best parameters
    umap_model = UMAP(
        n_neighbors=int(best_params["n_neighbors"]),
        n_components=int(best_params["n_components"]),
        min_dist=float(best_params["min_dist"]),
        metric="cosine",
        random_state=random_seed,
        n_jobs=-1,
        low_memory=True,
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=int(best_params["min_cluster_size"]),
        min_samples=int(best_params["min_samples"]),
        metric="euclidean",
        cluster_selection_method="eom",
        core_dist_n_jobs=-1,
        prediction_data=True,
    )

    # Topic representation
    vectorizer_model = CountVectorizer(
        ngram_range=(1, 2), # What counts as a word in the vocabulary: Unigrams (1) & bigrams, aka. two-word phrases like "red pill" (2)
        stop_words=all_stops, # Exclude all stopwords defined above
        max_features=15_000, # Capvocabulary at the X most frequent terms
        min_df=3, # A term must appear in at least X documents to be included
        max_df=0.95, # A term appearing in more than X % of documents is excluded (excludes filler words/common terms not captured by the stopword list)
    )

    topic_model = BERTopic(
        embedding_model=None,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False, # Enable if you wish, but be warned, it is VERY slow.
        verbose=True,
    )

    # Training phase
    print("\n🔧 Training the model...")
    all_probs = []
    training_indices  = shuffle_indices[:actual_training_size]
    training_docs     = [docs[i] for i in training_indices]
    training_embeddings = embeddings[training_indices]  # reads from mmap

    topics_shuffled, probs = topic_model.fit_transform(training_docs, training_embeddings)
    all_probs.append(probs)
    topics_shuffled    = np.array(topics_shuffled)
    print(f"✅ Training complete – {len(np.unique(topics_shuffled))} unique topics "
          f"(including outlier -1)")

    # Prediction phase (only when corpus > training sample)
    if len(docs) > actual_training_size:
        remaining = len(docs) - actual_training_size
        print(f"\n🔮 Assigning topics to remaining {remaining:,} documents...")

        for i in tqdm(
            range(actual_training_size, len(docs), batch_size),
            desc="Predicting batches",
            leave=False,
        ):
            end_idx       = min(i + batch_size, len(docs))
            batch_indices = shuffle_indices[i:end_idx]
            batch_docs    = [docs[j] for j in batch_indices]
            batch_embs    = embeddings[batch_indices]

            batch_topics, batch_probs = topic_model.transform(batch_docs, batch_embs)
            topics_shuffled = np.concatenate([topics_shuffled, batch_topics])

    # Restore original document order
    topics = np.empty(len(topics_shuffled), dtype=topics_shuffled.dtype)
    topics[shuffle_indices] = topics_shuffled

    # Evaluation
    outlier_count = int((topics == -1).sum())
    n_topics      = len(np.unique(topics[topics != -1]))
    print(f"\n📊 Topics discovered  : {n_topics}")
    print(f"   Outlier rate       : {outlier_count / len(topics) * 100:.2f}% "
          f"({outlier_count:,} documents)")
    print(f"   Total documents    : {len(topics):,}")

    return topic_model, topics

### `fit_bertopic_with_outlier_handling`

Wraps `fit_bertopic` with a two-step outlier reduction:

- **Step 1** – main model fit via `fit_bertopic`
- **Step 2** – [Outlier reduction](https://maartengr.github.io/BERTopic/getting_started/outlier_reduction/outlier_reduction.html) in batches: each batch that contains outlier documents is passed to `topic_model.reduce_outliers(..., strategy="...")`

> Change `strategy` depending on your preferred method. 
> Currently using: `c-tf-idf`, calculates the c-TF-IDF representation for each outlier document and find the best matching c-TF-IDF topic representation using cosine similarity

After reduction, `topic_model.update_topics(docs, topics_reduced)` is called so that the model's internal topic representations and `get_topic_info()` counts reflect the *full corpus*, not just the training sample.

In [11]:
def fit_bertopic_with_outlier_handling(
    docs,
    embeddings,
    best_params,
    batch_size=50_000,
    training_sample_size=350_000,
    random_seed=42,
):
    overall_start = time.time()

    # Step 1: Main model
    print("=" * 70)
    print("STEP 1: Main Topic Modeling")
    print("=" * 70)

    topic_model, topics = fit_bertopic(
        docs, embeddings, best_params,
        batch_size=batch_size,
        training_sample_size=training_sample_size,
        random_seed=random_seed,
    )

    # Step 2: c-TF-IDF outlier reduction
    print("\n" + "=" * 70)
    print("STEP 2: Reducing Outliers")
    print("=" * 70)

    initial_outliers = int((topics == -1).sum())
    print(f"Outliers before reduction : {initial_outliers:,} "
          f"({initial_outliers / len(topics) * 100:.2f}%)")

    topics_reduced = topics.copy()

    for i in tqdm(
        range(0, len(docs), batch_size),
        desc="Outlier reduction",
    ):
        end_idx      = min(i + batch_size, len(docs))
        batch_docs   = docs[i:end_idx]
        batch_topics = topics[i:end_idx]

        # Skip batches with no outliers (avoids unnecessary computation)
        if not (batch_topics == -1).any():
            continue

        reduced = topic_model.reduce_outliers(
            batch_docs,
            batch_topics,
            strategy="c-tf-idf",
        )
        topics_reduced[i:end_idx] = reduced

    final_outliers = int((topics_reduced == -1).sum())
    reduction      = initial_outliers - final_outliers
    print(f"Outliers after reduction  : {final_outliers:,} "
          f"({final_outliers / len(topics_reduced) * 100:.2f}%)")
    print(f"Reassigned                : {reduction:,} "
          f"({reduction / initial_outliers * 100:.1f}% of initial outliers)")

    # # Update topic representations to reflect full corpus
    # Without this, get_topic_info() counts only the training sample.
    print("\n🔄 Updating topic representations...")
    topic_model.update_topics(docs, topics=topics_reduced)
    print("✅ Topic representations updated")

    # Final summary
    n_topics = len(np.unique(topics_reduced[topics_reduced != -1]))
    print("\n" + "=" * 70)
    print("FINAL RESULTS")
    print("=" * 70)
    print(f"Total topics              : {n_topics}")
    print(f"Final outlier rate        : {final_outliers / len(topics_reduced) * 100:.2f}% "
          f"({final_outliers:,} documents)")

    elapsed = time.time() - overall_start
    h, rem  = divmod(int(elapsed), 3600)
    m, s    = divmod(rem, 60)
    print(f"Total processing time     : {h}h {m}m {s}s")

    return topic_model, topics_reduced

---
# 4. Fit model & reduce outliers

This cell runs both steps in sequence:
1. Fits BERTopic on the training sample and assigns topics to the full corpus.
2. Reduces outliers in batches..
3. Updates the model's internal representations to reflect the full corpus.

In [ ]:
# ⚠️ Runtime warning: varies for hardware & sample configurations.
topic_model_final, topics_final = fit_bertopic_with_outlier_handling(
    docs=preprocessed_docs,
    embeddings=embeddings,
    best_params=best_params,
    batch_size=BATCH_SIZE,
    training_sample_size=ACTUAL_TRAINING_SIZE,
    random_seed=RANDOM_SEED,
)

In [ ]:
# Validation
print(f"   Topics generated         : {len(topic_model_final.get_topic_info()):,}")
print(f"   Documents assigned       : {len(topics_final):,}")

non_outlier = topics_final[topics_final != -1]
print(f"   Non-outlier documents    : {len(non_outlier):,} "
      f"({len(non_outlier) / len(topics_final) * 100:.1f}%)")

# Silhouette score
print("\n📊 Calculating Silhouette Score (outliers excluded)...")
sil_n      = min(100_000, len(topics_final)) # Set minimum sample size for calculation
sil_idx    = np.random.choice(len(topics_final), size=sil_n, replace=False)
sil_topics = topics_final[sil_idx]
sil_embs   = embeddings[sil_idx]
mask       = sil_topics != -1

if mask.sum() > 1 and len(np.unique(sil_topics[mask])) > 1:
    sil = silhouette_score(sil_embs[mask], sil_topics[mask])
    print(f"   Silhouette (n={mask.sum():,}) : {sil:.4f}")
else:
    print("   ⚠️  Not enough clusters for silhouette score")

## Save model & topic assignments

The model is saved in safetensors format (smaller, faster to load than the default pickle). The topic assignments array and a results CSV (doc_id + text + topic) are saved alongside it.

In [ ]:
print("💾 Saving outputs...\n")

# Topic model
topic_model_final.save(
    str(TOPIC_MODEL_PATH),
    serialization="safetensors",
    save_ctfidf=True,
)
print(f"✅ Model saved.")

# Topics array
np.save(TOPICS_ASSIGNED_PATH, topics_final)
print(f"✅ Topics saved.")

# Results CSV (doc_id + text + topic)
results_df = pd.DataFrame({
    "doc_id": preprocessed_ids,
    "preprocessed_text": preprocessed_docs,
    "topic": topics_final,
})
results_df.to_csv(TOPIC_MODEL_RESULTS_PATH, index=False, compression="gzip")
print(f"✅ Results CSV saved.")

# Topic info
topic_info = topic_model_final.get_topic_info()
topic_info.to_csv(TOPIC_INFO_PATH, index=False)
print(f"✅ Topic info saved.")

# Excel version
xl_path = TOPIC_INFO_PATH.with_suffix(".xlsx")
with pd.ExcelWriter(xl_path, engine="openpyxl") as writer:
    topic_info_out = topic_info.copy()
    topic_info_out.insert(0, "Category", "") # For manually coding categories (overarching the topics)
    topic_info_out.insert(0, "Topic Label", "") # For manually coding topic labels
    topic_info_out.to_excel(writer, index=False, sheet_name="Topic Info")
    ws = writer.sheets["Topic Info"]
    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        ws.column_dimensions[col[0].column_letter].width = min(max_len + 4, 80)
print(f"✅ Topic info XL saved.")

print("\n✅ All outputs saved.")

---
## ✅ Summary & next steps

| Output | Description |
|--------|-------------|
| Topic model | Fitted BERTopic model (safetensors) |
| Topic assignments | Array of topic IDs, one per document |
| Results CSV | doc_id + text + topic |
| Topic info | Topic ID, size, top words, counts |


**Next steps:** Label topics in `topic_info.xlsx` (human or LLM-assisted), then load `bertopic_results_final.csv.gz` for downstream content analysis.|

---
# AI disclosure statement

AI tools were used to assist:
- developing, labelling, and debugging code
- formatting Markdown cells

AI tools used:
- [CursorAI (Desktop version)](https://cursor.com/agents)
- [Claude AI](https://claude.ai/)
- [ChatGPT](https://chatgpt.com/)

I acknowledge my responsibility as a researcher to thoroughly verify all outputs and content produced by AI tools and accept full accountability for their accuracy and validity.

XXX

---
# References
Grootendorst, M. (2022). BERTopic: Neural topic modeling with a class-based TF-IDF procedure. [arXiv preprint arXiv:2203.05794](https://arxiv.org/abs/2203.05794).